# 06 — rough Bergomi calibration

Calibrate (H, eta, rho, xi0) to the same SPX surface used for Heston. Expected: comparable RMSE overall, often tighter at short maturities — that is the empirical fingerprint of roughness.

## Context

rBergomi has no closed-form characteristic function, so every objective evaluation is a Monte Carlo simulation. The MC seed is fixed across the optimizer call so the objective is a *deterministic* function of $(H, \eta, \rho, \xi_0)$ — this is essential for L-BFGS-B's finite-difference gradients.

We typically achieve IV RMSE comparable to Heston on the same surface, sometimes slightly better at the short end — but the real diagnostic is the *term structure of ATM skew* in notebook 07, where rBergomi pulls clearly ahead.

In [ ]:
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

from volengine.backtesting import load_option_chain, filter_for_calibration
from volengine.calibration import IVQuote, calibrate_rbergomi
from volengine.surfaces import implied_vol
from volengine.models.rbergomi import rbergomi_price

In [ ]:
snapshot = load_option_chain('SPY', dt.date.today(), provider='yfinance')
filtered = filter_for_calibration(snapshot, moneyness_band=(0.9, 1.1))
quotes = []
for _, row in filtered.iterrows():
    iv = implied_vol(row['mid'], snapshot.spot, row['strike'],
                     row['dte_years'], snapshot.r, snapshot.q, 'call')
    if np.isfinite(iv):
        quotes.append(IVQuote(K=row['strike'], T=row['dte_years'], iv_mkt=iv, weight=1.0))
print(f'{len(quotes)} quotes feed calibration.')
result = calibrate_rbergomi(quotes, S0=snapshot.spot, r=snapshot.r, q=snapshot.q,
                            n_paths=15_000, n_steps_per_year=80)
result

## Calibrated parameters

SPX-typical rBergomi parameters cluster around $H \in [0.07, 0.15]$, $\eta \in [1.5, 2.5]$, $\rho \in [-0.95, -0.7]$. The $\xi_0$ parameter, in this simplified setup, is a flat level; a production calibration would fit a piecewise-constant forward variance curve from ATM term structure.

## Fit overlay

**Figure.** Market IVs versus calibrated rBergomi IVs across maturities. Comparable in-sample to Heston on the same surface — out-of-sample superiority shows up in the skew term structure.